# Create AAV6-ML plots for report for internship in AG Grimm
Author: Kolja Hildenbrand

Created: 01/04/2026

## 1. Packages

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import re
import matplotlib as plt
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import product
from matplotlib_venn import venn2

## 2. Create Paths

In [ ]:
#---------------------------------
#---------------------------------
#---------------------------------

#general Path

general_dir = Path('/Users/kollybook/Library/Mobile Documents/com~apple~CloudDocs/Kolja_Hildenbrand/Uni/Master_Infectious_Diseases/Grimm_internship/Report_Grimm/Data') # <----- Hier den Server Path anpassen
os.makedirs(general_dir, exist_ok=True)

#---------------------------------
#---------------------------------
#---------------------------------

# Path for NGS data
NGS_dir = general_dir/'NGS_data'
os.makedirs(NGS_dir, exist_ok=True)

# Path for tables
tables_dir = general_dir/'Tables'
os.makedirs(tables_dir, exist_ok=True)

#Path for plots
plots_dir = general_dir/'Figures'
os.makedirs(plots_dir, exist_ok=True)

## 3. Define Functions
### 3.1. Helper functions

In [ ]:
# read csv files
def read_csv_file (path, file_name):
    df = pd.read_csv(path / f"{file_name}.csv")
    return df

In [ ]:
# extract list information from table
def extract_info(table, *columns):
    results = []
    
    for col in columns:
        if col not in table.columns:
            raise ValueError(f"Column '{col}' not found in table")
        
        unique_vals = (
            table[col]
            .dropna()
            .unique()
            .tolist()
        )
        
        results.append(sorted(unique_vals))
    
    return tuple(results)

In [ ]:
def sort_file_names (name_list):
    FILE_NAMES = {
        "gDNA": {
            "heart": {"biological": [], "technical": []},
            "liver": {"biological": [], "technical": []}
        },
        "cDNA": {
            "heart": {"biological": [], "technical": []},
            "liver": {"biological": [], "technical": []}
        }
    }
    
    for name in name_list:
        parts = name.split("_")
        n_parts = len(parts)
    
        # ext type
        if name.startswith("gDNA"):
            dna = "gDNA"
        elif name.startswith("cDNA"):
            dna = "cDNA"
        else:
            continue
    
        # Tissue 
        tissue = parts[1]
    
        if tissue not in ["heart", "liver"]:
            continue  
    
        # bio or tech
        if n_parts == 3:
            category = "biological"
        elif n_parts == 4:
            category = "technical"
        else:
            continue
    
        FILE_NAMES[dna][tissue][category].append(name)

    return FILE_NAMES 

### 3.2. Script Functions

In [ ]:
input_lib = dict_library['liver']['gDNA']
df = df_long_table[
        (df_long_table["Tissue"] == 'liver') &
        (df_long_table["Extraction_type"] == 'gDNA')
    ][['AA_sequence', 'Mouse_ID', 'Proportion', 'Cum_prop', 'Log2_enrichment', 'Pseudo']].copy()

In [ ]:
input_lib["Data"] = 'input_library'
df = df.rename(columns = {'Mouse_ID': 'Data'})

In [ ]:
display (input_lib, df)

In [ ]:
combined = pd.concat([input_lib, df], axis=0, ignore_index=True)
combined

In [ ]:
# Function to create Histogram of sample proportion vs input proportion

def distribution_histogram (table, tissue, extraction, column, save=False, output_path=None):
    
    x_min = table[column].min() 
    x_max = table[column].max()
    
    input_library = dict_library[tissue][extraction].copy()
    df = table[
        (table["Tissue"] == tissue) &
        (table["Extraction_type"] == extraction)
    ][['AA_sequence', 'Mouse_ID', 'Proportion', 'Cum_prop', 'Log2_enrichment', 'Pseudo']].copy()
    df = df.rename(columns = {'Mouse_ID': 'Data'})
    df = pd.concat([input_lib, df], axis=0, ignore_index=True)

    df = df[df[column].notna()].copy()

    bin_edges = np.logspace(np.log10(x_min), np.log10(x_max), 100)

    sns.set_style("white")
    plt.rcParams.update({
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "legend.fontsize": 10,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10
    })

    fig, ax = plt.subplots(figsize=(7.5, 5.2))
    
    present_data = df['Data'].unique()
    for data in present_data:
        sub = df[df["Data"] == data]

        ax.hist(
            sub[column],
            bins=bin_edges,
            alpha=0.35,
            label=data,
            edgecolor="none"
        )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(x_min, x_max)
    
    ax.set_xlabel("Variant proportion in sample")
    ax.set_ylabel("Number of AA sequences")
    ax.set_title(f"{extraction} {tissue}: distribution of AA-sequence proportions")

    # cleaner legend
    ax.legend(title="Mouse ID", frameon=False, ncol=2)

    
    sns.despine()
    plt.tight_layout()

    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        fig.savefig(output_path, dpi=300, bbox_inches="tight")

    plt.show()

In [ ]:
# Function to create Log2_enrichment distribution ecdf figure of one tissue/ext type with hue between mouse_ID

def distribution_ecdf(table, tissue, extraction, column, save=False, output_path=None):

    x_min = table[column].min() 
    x_max = table[column].max()
    
    df = table[
        (table["Tissue"] == tissue) &
        (table["Extraction_type"] == extraction)
    ].copy()

    df = df[df[column].notna()].copy()
    
    sns.set_style("white")
    plt.rcParams.update({
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "legend.fontsize": 10
    })

    fig, ax = plt.subplots(figsize=(7.5, 5.2))

    sns.ecdfplot(
        data=df,
        x=column,
        hue="Mouse_ID",
        ax=ax,
        linewidth=2
    )
    
    # reference line no enrichment
    ax.axvline(
        0, 
        linestyle="--", 
        linewidth=1.5, color="red", alpha=0.9
    )
    
    mean_val = df[column].mean()
    ax.axvline(
        mean_val, 
        linestyle="--", 
        linewidth=1.5, 
        color="blue", 
        alpha=0.9
    )
        
    ax.set_xlim(x_min, x_max)
    
    ax.set_xlabel(f"Variant {column}")
    ax.set_ylabel("Cumulative fraction of AA sequences")
    ax.set_title(f"{extraction} {tissue}\n cumulative distribution of {column}\n Mean = {mean_val:.2f}")

    ax.legend(title="Mouse ID", loc="upper right", frameon=False, ncol=2)
    
    sns.despine()
    plt.tight_layout()

    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        fig.savefig(output_path, dpi=300, bbox_inches="tight")

    plt.show()



In [ ]:
def distribution_kde(table, tissue, extraction, column, save=False, output_path=None):

    input_library = dict_library[tissue][extraction].copy()

    df = table[
        (table["Tissue"] == tissue) &
        (table["Extraction_type"] == extraction)
    ][['AA_sequence', 'Mouse_ID', 'Proportion', 'Cum_prop', 'Log2_enrichment', 'Pseudo']].copy()

    df = df.rename(columns={'Mouse_ID': 'Data'})

    # 👉 concat statt merge
    df = pd.concat([input_library, df], axis=0, ignore_index=True)

    # remove NaN + zeros (wichtig für log!)
    df = df[df[column].notna()].copy()
    df = df[df[column] > 0].copy()

    # 👉 log-transform für KDE
    df["log_value"] = np.log10(df[column])

    sns.set_style("white")
    plt.rcParams.update({
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "legend.fontsize": 10
    })

    fig, ax = plt.subplots(figsize=(7.5, 5.2))

    # KDE plot
    sns.kdeplot(
        data=df,
        x="log_value",
        hue="Data",
        common_norm=False,   # wichtig für Vergleich!
        fill=True,
        alpha=0.3,
        linewidth=1.5,
        ax=ax
    )

    # Achsen zurück in log-space beschriften
    xticks = np.arange(np.floor(df["log_value"].min()), np.ceil(df["log_value"].max()))
    ax.set_xticks(xticks)
    ax.set_xticklabels([f"$10^{{{int(x)}}}$" for x in xticks])

    ax.set_xlabel("Variant proportion")
    ax.set_ylabel("Density")
    ax.set_title(f"{extraction} {tissue}: distribution of AA-sequence proportions (KDE)")

    ax.legend(title="Data", frameon=False)

    sns.despine()
    plt.tight_layout()

    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        fig.savefig(output_path, dpi=300, bbox_inches="tight")

    plt.show()

## 4. Script
### 4.1. Load CSV Files

In [ ]:
%%time
#load long table
df_long_table = read_csv_file(tables_dir / 'summary', "df_long_combined_bio_tech")

# load pooled table
df_pooled = read_csv_file(tables_dir / 'summary', "df_pooled_table")

In [ ]:
print ('Long Table')
display (df_long_table)

print ('Pooled Table')
display (df_pooled)

### 4.2. Extract information from table

In [ ]:
SAMPLE, EXT, TISSUE, SEX, MOUSE_ID = extract_info(
    df_long_table, 
    'Sample', 
    'Extraction_type',
    "Tissue", 
    'Sex', 
    'Mouse_ID'
)

In [ ]:
# Sort different file names in extraction and biological or technical replicates
DICT_NAMES = sort_file_names (SAMPLE)

In [ ]:
DICT_NAMES

### 4.3. Load tissue/ext specific librarys

In [ ]:
%%time
# Load tissue specific library 
dict_library = {}
for tissue, ext in product(TISSUE, EXT):
    df = read_csv_file(tables_dir/tissue, f'df_input_library_for_{ext}_{tissue}')
    dict_library.setdefault(tissue,{})[ext] = df

# Load raw library  for corr with special library
df_raw_input =  read_csv_file(tables_dir, 'df_input_lib_raw')

### 4.4. Comparison between input and samples regarding proportion

#### 4.4.1. Histogram with am propurtion with Hue = Sample


In [ ]:
distribution_histogram (df_long_table, 'heart', 'cDNA', 'Proportion')

#### 4.4.2. ECDF with samples and input

#### 4.4.3. Venn2 Input vs mean proportion in sample

### 4.5. ECDF Log2 distribution between samples

In [ ]:
%%time
for tissues, exts in product (TISSUE, EXT):
    distribution_ecdf (df_long_table, tissues, exts, 'Log2_enrichment');

### 4.6. Comparison functional selection between RNA and DNA samples

#### 4.6.1. Violin plot mean between biological replicate

#### 4.6.2. KDE plot for liver and heart (DNA and RNA in one plot, x = enrichment, y = n variant)

#### 4.6.3. Venn2 plot with top 10000 enrichment
    - to show already the selection difference between biological step and tissue

#### 4.6.4. Diagramm of Venn2 plot overlap between top 1 and top 10^6